<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/7_2_Principal_Component_Analysis_(PCA)_and_SVD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VII. Dimensionality Reduction, Unsupervised Learning, and Geometry of Data

## 1. Curse and Blessings of Dimensionality    (Geometry of the d‑dimensional sphere and cube, concentration of measure, condition number of random matrices)
## 2. Principal Component Analysis (PCA) and SVD
## 3. Nonlinear Dimensionality Reduction: t‑SNE, UMAP, Isomap
## 4. Clustering: K‑means, Hierarchical, DBSCAN


## Интуиция и геометрия метода главных компонент

### 1. Проблема многомерных данных

Человеческое восприятие ограничено тремя измерениями. Мы легко строим диаграммы рассеяния для двух или трёх признаков, но как быть, если их сотни или тысячи? Такие ситуации типичны: анализ экспрессии генов (тысячи генов), пиксели изображений, текстовые данные в виде мешка слов. Прямая визуализация невозможна, а многие алгоритмы машинного обучения страдают от «проклятия размерности». Хочется сжать данные, сохранив при этом как можно больше полезной информации.

**Метод главных компонент (Principal Component Analysis, PCA)** — один из самых элегантных и широко применяемых способов снижения размерности. Его идея: найти в многомерном пространстве небольшое число направлений, вдоль которых данные меняются сильнее всего, и спроецировать исходные точки на подпространство, натянутое на эти направления. Если данные действительно лежат вблизи некоторой низкоразмерной плоскости, такое проецирование почти не искажает их структуру, зато резко сокращает число признаков.

### 2. Направление максимальной дисперсии

Пусть у нас есть набор из $n$ точек $\mathbf{x}_1,\dots,\mathbf{x}_n \in \mathbb{R}^d$. Для простоты будем считать, что данные центрированы: среднее арифметическое по каждому признаку равно нулю (этого всегда можно добиться вычитанием среднего). Мы ищем прямую, проходящую через начало координат, задаваемую единичным вектором $\mathbf{w}$ ($\|\mathbf{w}\|=1$). Проекция точки $\mathbf{x}_i$ на эту прямую равна скалярному произведению $\mathbf{w}^T\mathbf{x}_i$. Дисперсия проекций

$$
\operatorname{Var}_{\mathbf{w}} = \frac{1}{n}\sum_{i=1}^n (\mathbf{w}^T\mathbf{x}_i)^2
= \mathbf{w}^T\!\left(\frac{1}{n}\sum_{i=1}^n \mathbf{x}_i\mathbf{x}_i^T\right)\!\mathbf{w}
= \mathbf{w}^T \mathbf{\Sigma} \mathbf{w},
$$

где $\mathbf{\Sigma} = \frac{1}{n}\mathbf{X}^T\mathbf{X}$ — выборочная ковариационная матрица размера $d\times d$.

Задача PCA для первой компоненты: найти $\mathbf{w}$, максимизирующий $\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}$ при условии $\|\mathbf{w}\|=1$. Это классическая задача на собственные значения: решением является собственный вектор, отвечающий **наибольшему** собственному числу $\lambda_1$ матрицы $\mathbf{\Sigma}$. Сама дисперсия проекций при этом равна $\lambda_1$.

### 3. Эллипсоид рассеяния и собственные векторы

Ковариационная матрица $\mathbf{\Sigma}$ симметрична и положительно полуопределена. Она задаёт эллипсоид рассеяния данных:

$$
\{ \mathbf{x} \mid \mathbf{x}^T \mathbf{\Sigma}^{-1} \mathbf{x} = \text{const} \}.
$$

Оси этого эллипсоида направлены вдоль собственных векторов $\mathbf{v}_1,\dots,\mathbf{v}_d$ матрицы $\mathbf{\Sigma}$, а их длины пропорциональны $\sqrt{\lambda_i}$. Первая главная компонента — это направление самой длинной оси эллипсоида, т.е. собственный вектор с наибольшим $\lambda$. Вторая компонента — следующая по длине ось, ортогональная первой, и так далее.

Геометрически PCA сводится к повороту исходной системы координат так, чтобы новые оси совпали с осями эллипсоида. Проекция на первые $k$ осей даёт наилучшую $k$-мерную аппроксимацию облака точек в смысле минимизации суммы квадратов расстояний от точек до подпространства.

### 4. Пример: рост и вес

Рассмотрим игрушечный набор данных: рост и вес 200 человек. Пусть рост распределён нормально со средним 170 см и стандартным отклонением 10 см, а вес линейно связан с ростом плюс шум.

Сгенерируем данные, вычислим ковариационную матрицу и её собственные векторы, нанесём на график первую главную компоненту.

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 200
height = np.random.normal(170, 10, n)
weight = 0.6 * height + np.random.normal(0, 5, n)

# Центрируем данные
X = np.column_stack([height, weight])
X_mean = X.mean(axis=0)
X_centered = X - X_mean

# Ковариационная матрица (смещённая оценка для PCA)
cov_matrix = np.cov(X_centered.T, bias=True)
eigvals, eigvecs = np.linalg.eigh(cov_matrix)

# Первая главная компонента — собственный вектор с максимальным λ
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]
pc1 = eigvecs[:, 0]

# Линия первой компоненты
line_x = np.linspace(X_centered[:,0].min(), X_centered[:,0].max(), 100)
line_y = (pc1[1] / pc1[0]) * line_x

plt.figure(figsize=(8,6))
plt.scatter(X_centered[:,0], X_centered[:,1], alpha=0.6, label='Данные (центрированные)')
plt.plot(line_x, line_y, 'r-', lw=2, label='Первая главная компонента')
plt.xlabel('Рост (центрированный)')
plt.ylabel('Вес (центрированный)')
plt.title('Рост и вес: первая главная компонента')
plt.legend()
plt.axis('equal')
plt.grid(True)
plt.show()

print("Собственные числа:", eigvals)
print("Первая компонента (вектор):", pc1)
print("Доля объяснённой дисперсии:", eigvals[0] / eigvals.sum())
```

Первая компонента улавливает основную закономерность: с увеличением роста вес в среднем растёт. Угол наклона прямой равен $\arctan(v_2/v_1)$, где $\mathbf{v}=(v_1,v_2)$ — собственный вектор. Вторая компонента (перпендикулярная) соответствует вариации веса при фиксированном росте — индивидуальным отклонениям от средней линии.

### 5. Поворот осей: главные компоненты как новые координаты

Если спроецировать центрированные данные на все главные компоненты, получим новые координаты

$$
\mathbf{Z} = \mathbf{X}_{\text{centered}} \mathbf{V},
$$

где столбцы матрицы $\mathbf{V}$ — собственные векторы $\mathbf{\Sigma}$. В этом базисе ковариационная матрица диагональна: $\frac{1}{n}\mathbf{Z}^T\mathbf{Z} = \operatorname{diag}(\lambda_1,\dots,\lambda_d)$. Это означает, что новые признаки (главные компоненты) некоррелированы. Более того, дисперсия $j$-го признака равна $\lambda_j$. Мы можем отбросить компоненты с малыми $\lambda_j$, теряя минимум информации.

Визуализируем повёрнутые данные для примера «рост–вес»:

```python
# Проекция на обе компоненты
Z = X_centered @ eigvecs

plt.figure(figsize=(8,6))
plt.scatter(Z[:,0], Z[:,1], alpha=0.6)
plt.xlabel('Главная компонента 1')
plt.ylabel('Главная компонента 2')
plt.title('Данные в пространстве главных компонент')
plt.axis('equal')
plt.grid(True)
plt.show()
```

Теперь облако вытянуто вдоль оси абсцисс — именно вдоль неё максимальная дисперсия.

### 6. Минимизация ошибки реконструкции

PCA можно вывести из другой, эквивалентной задачи: найти $k$-мерное линейное подпространство, минимизирующее сумму квадратов расстояний от точек до их проекций на это подпространство (ошибку реконструкции). Если $k=1$, ищется прямая, для которой сумма квадратов расстояний от точек до неё минимальна. Математически это сводится к той же проблеме собственных векторов, но с дополнительным интуитивным смыслом: мы хотим, чтобы сжатое представление $\mathbf{z}_i$ после обратного проецирования $\hat{\mathbf{x}}_i = \mathbf{V}_k \mathbf{z}_i$ (где $\mathbf{V}_k$ — первые $k$ собственных векторов) давало минимальное среднеквадратичное отклонение $\frac{1}{n}\sum_i \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2$.

Таким образом, PCA — это не просто «поворот и отрезание», а оптимальный линейный метод сжатия информации с точки зрения сохранения структуры данных.

### 7. Резюме

Метод главных компонент находит новые ортогональные оси, упорядоченные по убыванию дисперсии. Первая ось — направление максимального разброса, вторая — максимального среди оставшихся, перпендикулярного первой, и т.д. Геометрически это сводится к анализу эллипсоида рассеяния, оси которого — собственные векторы ковариационной матрицы. Проецируя данные на первые несколько осей, мы получаем сжатое представление, минимизирующее среднеквадратичную ошибку восстановления.

**Упражнения для читателя:**
1. Почему перед PCA необходимо центрировать данные? Что произойдёт, если этого не сделать?
2. Как изменится первая главная компонента, если рост измерять в метрах, а вес — в фунтах? Как избежать зависимости PCA от масштаба признаков?
3. Докажите, что сумма собственных чисел ковариационной матрицы равна сумме дисперсий исходных признаков. Как это свойство используется для выбора числа компонент?

## Реализация PCA с нуля на NumPy

В предыдущем разделе мы обсудили интуитивный смысл метода главных компонент. Теперь мы перейдём к практической реализации PCA «с нуля», используя только NumPy. Это позволит глубже понять алгоритм, его численные аспекты и подготовит почву для более стабильных методов, основанных на сингулярном разложении.

### 1. Алгоритм по шагам

Пусть дана матрица данных $\mathbf{X} \in \mathbb{R}^{n \times d}$, где $n$ — число наблюдений, $d$ — число признаков. Требуется получить матрицу проекций $\mathbf{Z} \in \mathbb{R}^{n \times k}$ на первые $k$ главных компонент.

**Шаг 1. Центрирование.**  
Вычтем из каждого признака его среднее значение:

$$
\mathbf{X}_{\text{centered}} = \mathbf{X} - \mathbf{1}_n \boldsymbol{\mu}^T, \qquad
\boldsymbol{\mu} = \frac{1}{n} \sum_{i=1}^n \mathbf{x}_i.
$$

Где $\mathbf{1}_n$ — вектор из единиц длины $n$. Центрирование гарантирует, что данные имеют нулевое среднее, что необходимо для корректной работы PCA.

**Шаг 2. Ковариационная матрица.**  
Вычислим выборочную ковариационную матрицу (несмещённая оценка):

$$
\mathbf{C} = \frac{1}{n-1} \mathbf{X}_{\text{centered}}^T \mathbf{X}_{\text{centered}} \;\in\; \mathbb{R}^{d \times d}.
$$

Элемент $C_{ij}$ — ковариация между $i$-м и $j$-м признаками.

**Шаг 3. Собственные значения и векторы.**  
Решаем спектральную задачу:

$$
\mathbf{C} \mathbf{v}_j = \lambda_j \mathbf{v}_j, \quad j=1,\dots,d.
$$

Поскольку $\mathbf{C}$ симметрична и положительно полуопределена, все $\lambda_j \ge 0$, а векторы $\mathbf{v}_j$ можно выбрать ортонормированными.

**Шаг 4. Сортировка.**  
Упорядочим собственные значения по убыванию: $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_d \ge 0$ и соответствующие им векторы.

**Шаг 5. Выбор первых $k$ компонент.**  
Образуем матрицу $\mathbf{W}_k \in \mathbb{R}^{d \times k}$, столбцы которой — первые $k$ собственных векторов. Они образуют ортонормированный базис искомого подпространства.

**Шаг 6. Проекция.**  
Новые координаты (главные компоненты) получаются умножением центрированных данных на матрицу весов:

$$
\mathbf{Z} = \mathbf{X}_{\text{centered}} \mathbf{W}_k \;\in\; \mathbb{R}^{n \times k}.
$$

Обратное преобразование: $\hat{\mathbf{X}} = \mathbf{Z} \mathbf{W}_k^T + \boldsymbol{\mu}$, которое восстанавливает данные с минимальной среднеквадратичной ошибкой среди всех линейных $k$-мерных приближений.

### 2. Реализация на Python

Напишем функцию `my_pca`, реализующую описанный алгоритм.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import time

def my_pca(X, k):
    """
    Простой PCA через собственные векторы ковариационной матрицы.
    
    Параметры:
        X : ndarray, shape (n, d)
            Исходные данные (без центрирования).
        k : int
            Число главных компонент (k <= d).
    
    Возвращает:
        X_pca : ndarray, shape (n, k)
            Проекция данных на первые k главных компонент.
        components : ndarray, shape (k, d)
            Главные компоненты (векторы-строки).
        explained_variance : ndarray, shape (k,)
            Дисперсия каждой из k компонент.
        explained_variance_ratio : ndarray, shape (k,)
            Доля объяснённой дисперсии каждой компонентой.
        mean : ndarray, shape (d,)
            Средние значения признаков, использованные для центрирования.
    """
    # Шаг 1: Центрирование
    mean = np.mean(X, axis=0)
    X_centered = X - mean

    # Шаг 2: Ковариационная матрица (несмещённая оценка)
    n = X.shape[0]
    C = (X_centered.T @ X_centered) / (n - 1)

    # Шаг 3: Спектральное разложение
    eig_vals, eig_vecs = np.linalg.eigh(C)  # eigh для симметричных матриц

    # Шаг 4: Сортировка по убыванию собственных значений
    # eigh возвращает в возрастающем порядке, поэтому развернём
    idx = np.argsort(eig_vals)[::-1]
    eig_vals = eig_vals[idx]
    eig_vecs = eig_vecs[:, idx]

    # Шаг 5: Выбор первых k компонент (векторы-столбцы → строки)
    W_k = eig_vecs[:, :k]          # shape (d, k)
    components = W_k.T             # shape (k, d)

    # Шаг 6: Проекция
    X_pca = X_centered @ W_k       # shape (n, k)

    # Вычисление объяснённой дисперсии
    explained_variance = eig_vals[:k]
    explained_variance_ratio = explained_variance / np.sum(eig_vals)

    return X_pca, components, explained_variance, explained_variance_ratio, mean
```

**Замечание о знаках:** Собственные векторы определены с точностью до знака: если $\mathbf{v}$ — собственный вектор, то $-\mathbf{v}$ также собственный вектор. Поэтому знаки главных компонент могут быть противоположными в разных реализациях — это абсолютно нормально и не влияет на геометрический смысл.

### 3. Демонстрация на синтетических данных

Создадим набор данных со скрытой низкоразмерной структурой и коррелированными признаками.

```python
np.random.seed(42)
n, d = 100, 3
# Генерируем скрытые факторы (2 компоненты)
Z_true = np.random.randn(n, 2)
# Случайная матрица нагрузок
W_true = np.random.randn(2, d)
# Данные = факторы * нагрузки + небольшой шум
X = Z_true @ W_true + 0.2 * np.random.randn(n, d)

# Применяем нашу реализацию PCA с k=2
k = 2
X_pca, comps, var, var_ratio, mean = my_pca(X, k)

print("Собственные значения:", var)
print("Доля объяснённой дисперсии:", var_ratio)
print("Кумулятивная доля:", np.sum(var_ratio))
print("Первые две главные компоненты (строки):\n", comps)

# Сравним с sklearn
pca_sk = PCA(n_components=k)
X_pca_sk = pca_sk.fit_transform(X)

print("\nsklearn explained_variance_ratio_:", pca_sk.explained_variance_ratio_)
print("sklearn components:\n", pca_sk.components_)

# Проверим совпадение с точностью до знака
# Абсолютные значения корреляции между компонентами должны быть близки к 1
for i in range(k):
    corr = np.corrcoef(X_pca[:, i], X_pca_sk[:, i])[0, 1]
    print(f"Корреляция между PC{i+1} (моя) и PC{i+1} (sklearn): {corr:.6f}")
```

Вы увидите, что доли объяснённой дисперсии совпадают, а корреляция между соответствующими компонентами равна ±1 с высокой точностью — наша реализация корректна.

### 4. Визуализация и анализ объяснённой дисперсии

Построим график доли объяснённой дисперсии и кумулятивной суммы. Это помогает выбрать $k$.

```python
# Полный PCA (k=d) для анализа дисперсии
_, _, var_all, var_ratio_all, _ = my_pca(X, d)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.bar(range(1, d+1), var_ratio_all, alpha=0.7, label='Индивидуальная')
plt.step(range(1, d+1), np.cumsum(var_ratio_all), where='mid', label='Кумулятивная')
plt.xlabel('Номер главной компоненты')
plt.ylabel('Доля объяснённой дисперсии')
plt.title('Объяснённая дисперсия')
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Данные в пространстве первых двух компонент')
plt.axis('equal')
plt.grid(True)

plt.tight_layout()
plt.show()
```

Из графика видно, что первые две компоненты объясняют почти всю дисперсию — это согласуется с тем, что данные были порождены двумерным латентным процессом.

### 5. Сравнение времени выполнения с sklearn

```python
# Сравнение времени для большого набора данных (n=5000, d=100)
n_big, d_big = 5000, 100
X_big = np.random.randn(n_big, d_big)
k_big = 10

# Наша реализация
start = time.time()
X_pca_my, _, _, _, _ = my_pca(X_big, k_big)
time_my = time.time() - start

# sklearn
pca_sk_big = PCA(n_components=k_big)
start = time.time()
X_pca_sk_big = pca_sk_big.fit_transform(X_big)
time_sk = time.time() - start

print(f"Время выполнения (my_pca): {time_my:.4f} сек")
print(f"Время выполнения (sklearn): {time_sk:.4f} сек")
```

`sklearn` обычно быстрее благодаря использованию SVD и оптимизированных библиотек (LAPACK). Наивный метод через ковариационную матрицу требует $O(d^3)$ на собственные векторы, что может быть дорого при больших $d$.

### 6. Ограничения наивной реализации и переход к SVD

Наш код имеет два серьёзных недостатка.

**Высокая размерность ($d > n$).**  
Если признаков больше, чем наблюдений (типично для геномики, текстов), ковариационная матрица $\mathbf{C}$ имеет размер $d \times d$ и её собственное разложение обходится очень дорого. Вместо этого выгоднее работать с матрицей Грама $\mathbf{G} = \mathbf{X}_{\text{centered}} \mathbf{X}_{\text{centered}}^T$ размера $n \times n$, собственные векторы которой связаны с искомыми компонентами. Ещё лучше — использовать SVD.

**Численная неустойчивость.**  
При малых собственных значениях процедура `eig` может давать отрицательные значения из-за ошибок округления. SVD свободен от этой проблемы и является стандартом де-факто для PCA.

**Связь с SVD** (анонс следующего раздела).  
Пусть $\mathbf{X}_{\text{centered}} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$ — усечённое сингулярное разложение, где $\mathbf{U} \in \mathbb{R}^{n \times r}$, $\boldsymbol{\Sigma} \in \mathbb{R}^{r \times r}$, $\mathbf{V} \in \mathbb{R}^{d \times r}$, $r = \min(n,d)$. Тогда:

$$
\mathbf{C} = \frac{1}{n-1} \mathbf{V} \boldsymbol{\Sigma}^2 \mathbf{V}^T,
$$

то есть $\mathbf{V}$ — собственные векторы, а $\lambda_j = \sigma_j^2 / (n-1)$. Проекция: $\mathbf{Z} = \mathbf{U}_k \boldsymbol{\Sigma}_k$. Таким образом, PCA можно выполнить через SVD центрированной матрицы данных, не строя ковариационную матрицу, что значительно стабильнее и эффективнее. Именно так работают `sklearn.decomposition.PCA` и большинство серьёзных библиотек.

### 7. Важное предупреждение: центрирование vs масштабирование

Перед применением PCA данные необходимо **центрировать** (вычесть среднее), иначе первый компонент может уйти в направление среднего, а не вариации. Однако **масштабирование** (деление на стандартное отклонение) не является обязательной частью PCA как такового. Если признаки измеряются в разных единицах (например, возраст в годах и доход в тысячах долларов), то признаки с бо́льшим численным размахом будут доминировать в ковариационной матрице, искажая смысл главных компонент. В таких случаях применяют **стандартизацию** (центрирование + масштабирование до единичной дисперсии), что эквивалентно PCA на корреляционной матрице. Встроенного рецепта нет: выбор зависит от природы данных и целей анализа.

---

### Резюме

Мы построили работающую реализацию PCA «с нуля» на NumPy, следуя классическому алгоритму: центрирование, ковариационная матрица, спектральное разложение, отбор $k$ компонент, проекция. Код протестирован на синтетических данных и сопоставлен с эталонной библиотекой `sklearn`. Проанализирована объяснённая дисперсия, указаны численные и вычислительные ограничения подхода, мотивирующие переход к SVD-реализации, которая будет рассмотрена в следующем разделе.

**Упражнения для читателя:**
1. Почему в формуле ковариационной матрицы мы использовали $n-1$, а не $n$? Изменится ли что-то в PCA, если использовать смещённую оценку?
2. Реализуйте функцию `my_pca_svd(X, k)`, использующую `np.linalg.svd` для центрированной матрицы, и сравните точность с `my_pca` на данных, где $d \approx n$.
3. Исследуйте, как меняются главные компоненты, если перед PCA признаки стандартизировать. В каком случае это необходимо?

## PCA через сингулярное разложение: численная устойчивость и эффективность

Мы уже реализовали PCA «в лоб» — через ковариационную матрицу и спектральное разложение. Однако этот подход страдает от вычислительной неэффективности при большом числе признаков и численной неустойчивости, когда собственные значения близки к нулю. Промышленный стандарт — использование сингулярного разложения (SVD) центрированной матрицы данных. В этом разделе мы покажем, почему SVD предпочтительнее, подтвердим это кодом и дадим практические рекомендации.

### 1. Краткое напоминание SVD

Для любой вещественной матрицы $\mathbf{X} \in \mathbb{R}^{n \times d}$ ранга $r \le \min(n,d)$ **сингулярное разложение** (Singular Value Decomposition) имеет вид

$$
\mathbf{X} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T,
$$

где
* $\mathbf{U} \in \mathbb{R}^{n \times r}$ — матрица левых сингулярных векторов, $\mathbf{U}^T\mathbf{U} = \mathbf{I}_r$,
* $\mathbf{V} \in \mathbb{R}^{d \times r}$ — матрица правых сингулярных векторов, $\mathbf{V}^T\mathbf{V} = \mathbf{I}_r$,
* $\boldsymbol{\Sigma} = \operatorname{diag}(\sigma_1,\dots,\sigma_r)$ — диагональная матрица сингулярных чисел, $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$.

Сингулярные числа $\sigma_i$ неотрицательны, их квадраты $\sigma_i^2$ — собственные значения как $\mathbf{X}^T\mathbf{X}$, так и $\mathbf{X}\mathbf{X}^T$.

### 2. Связь SVD с PCA

Пусть данные центрированы: $\mathbf{X}_{\text{centered}} \in \mathbb{R}^{n \times d}$, среднее по каждому столбцу равно нулю. Выборочная ковариационная матрица (несмещённая оценка) равна

$$
\mathbf{C} = \frac{1}{n-1} \mathbf{X}_{\text{centered}}^T \mathbf{X}_{\text{centered}} \;\in\; \mathbb{R}^{d \times d}.
$$

Подставим SVD центрированной матрицы $\mathbf{X}_{\text{centered}} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$:

$$
\mathbf{X}_{\text{centered}}^T \mathbf{X}_{\text{centered}} = \mathbf{V} \boldsymbol{\Sigma}^T \mathbf{U}^T \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T = \mathbf{V} \boldsymbol{\Sigma}^2 \mathbf{V}^T.
$$

Следовательно,

$$
\mathbf{C} = \frac{1}{n-1} \mathbf{V} \boldsymbol{\Sigma}^2 \mathbf{V}^T.
$$

Это спектральное разложение матрицы $\mathbf{C}$: столбцы $\mathbf{V}$ — собственные векторы, а соответствующие собственные значения равны $\lambda_i = \sigma_i^2 / (n-1)$. Таким образом, **правые сингулярные векторы $\mathbf{V}$ являются главными компонентами** (направлениями максимальной дисперсии), а сингулярные числа напрямую связаны с дисперсией вдоль этих направлений.

Проекция данных на первые $k$ главных компонент может быть получена без явного построения $\mathbf{C}$:

$$
\mathbf{Z} = \mathbf{X}_{\text{centered}} \mathbf{V}_k = \mathbf{U}_k \boldsymbol{\Sigma}_k,
$$

где $\mathbf{V}_k$ — первые $k$ столбцов $\mathbf{V}$, $\mathbf{U}_k$ — первые $k$ столбцов $\mathbf{U}$, $\boldsymbol{\Sigma}_k$ — верхний левый блок $\boldsymbol{\Sigma}$ размера $k \times k$. Это ключевое равенство: PCA можно выполнить, просто оставив первые $k$ компонент SVD.

### 3. Почему SVD предпочтительнее прямого вычисления собственных векторов $\mathbf{C}$

**Численная устойчивость.**  
Вычисление ковариационной матрицы $\mathbf{C}$ требует умножения $\mathbf{X}_{\text{centered}}^T \mathbf{X}_{\text{centered}}$, что может привести к потере точности, если матрица $\mathbf{X}$ плохо обусловлена (например, признаки почти линейно зависимы). Ошибки округления возводятся в квадрат. SVD, напротив, работает непосредственно с $\mathbf{X}_{\text{centered}}$ и не образует $\mathbf{C}$. Современные алгоритмы SVD (например, основанные на QR-разложении) чрезвычайно устойчивы и гарантируют неотрицательность сингулярных чисел. В случае `eig` малые собственные значения могут оказаться отрицательными из-за численных погрешностей, что нефизично для ковариационной матрицы.

**Эффективность для $d > n$ (больше признаков, чем наблюдений).**  
Прямое вычисление $\mathbf{C}$ требует $O(nd^2)$ операций и $O(d^2)$ памяти, а спектральное разложение — ещё $O(d^3)$. При $d \gg n$ это становится неприемлемым. С помощью SVD можно вычислить только правые сингулярные векторы через матрицу $\mathbf{X}_{\text{centered}}^T \mathbf{X}_{\text{centered}}$ или, что эффективнее, работать с левыми сингулярными векторами матрицы $\mathbf{X}_{\text{centered}} \mathbf{X}_{\text{centered}}^T$ размера $n \times n$, а затем получить правые. Библиотеки, как `sklearn`, используют усечённый SVD (`svd_solver='randomized'` или `'arpack'`), которые работают за $O(n d k)$.

**Отсутствие явного построения ковариационной матрицы.**  
Это не только экономит память, но и позволяет обрабатывать разреженные данные, не разрушая их структуру.

### 4. Реализация PCA через SVD на NumPy и сравнение времени

Напишем функцию `pca_svd`, использующую `np.linalg.svd` с параметром `full_matrices=False`. Сравним время работы с предыдущей `my_pca` на матрице $1000 \times 100$.

```python
import numpy as np
import time
import matplotlib.pyplot as plt

def pca_svd(X, k):
    """PCA через SVD центрированной матрицы."""
    mean = np.mean(X, axis=0)
    X_centered = X - mean
    # SVD с усечением (не вычислять полные матрицы U, V)
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    # Главные компоненты — строки Vt (первые k)
    components = Vt[:k]
    # Проекция = U_k * Sigma_k
    X_pca = U[:, :k] @ np.diag(S[:k])
    # Объяснённая дисперсия
    total_var = np.sum(S**2) / (X.shape[0] - 1)
    explained_variance = (S[:k]**2) / (X.shape[0] - 1)
    explained_variance_ratio = explained_variance / total_var
    return X_pca, components, explained_variance, explained_variance_ratio, mean

# Сравнение времени
n, d = 1000, 100
X = np.random.randn(n, d)
k = 10

# Старый метод через eig
def pca_eig(X, k):
    mean = np.mean(X, axis=0)
    Xc = X - mean
    C = (Xc.T @ Xc) / (X.shape[0] - 1)
    vals, vecs = np.linalg.eigh(C)
    idx = np.argsort(vals)[::-1]
    vecs = vecs[:, idx]
    vals = vals[idx]
    X_pca = Xc @ vecs[:, :k]
    return X_pca, vecs[:, :k].T, vals[:k], vals[:k]/np.sum(vals), mean

# Замер времени
start = time.time()
pca_eig(X, k)
t_eig = time.time() - start

start = time.time()
pca_svd(X, k)
t_svd = time.time() - start

print(f"Время PCA через eig: {t_eig:.4f} сек")
print(f"Время PCA через SVD: {t_svd:.4f} сек")
```

Результаты на типичном ноутбуке: SVD может быть быстрее или сопоставим при $d \le n$, но его главное преимущество — устойчивость и масштабируемость на высокие размерности.

### 5. Демонстрация численной неустойчивости `eig`

Создадим матрицу с почти коллинеарными столбцами: $\mathbf{X}_2 = \mathbf{X}_1 + 10^{-10} \cdot \text{шум}$. Из-за конечной точности вычислений ковариационная матрица может получить отрицательные собственные значения при использовании `eig`, тогда как SVD останется устойчивым.

```python
np.random.seed(0)
n = 100
X1 = np.random.randn(n)
X2 = X1 + 1e-10 * np.random.randn(n)  # почти линейно зависимы
X = np.column_stack([X1, X2])

# Центрируем
Xc = X - np.mean(X, axis=0)

# Метод eig
C = (Xc.T @ Xc) / (n - 1)
vals_eig, _ = np.linalg.eigh(C)
print("Собственные значения (eig):", vals_eig)

# Метод SVD
_, S, _ = np.linalg.svd(Xc, full_matrices=False)
vals_svd = S**2 / (n - 1)
print("Собственные значения из SVD:", vals_svd)
```

Одно из собственных значений от `eig` может оказаться отрицательным (например, $-10^{-16}$), в то время как SVD гарантирует неотрицательность $\sigma_i^2$. Это критично, когда собственные значения используются для определения ранга или объяснённой дисперсии: отрицательные значения ломают логику и могут привести к ошибкам в коде, если ожидается положительность.

### 6. Truncated SVD для огромных данных

Когда $n$ и $d$ велики, полное SVD-разложение слишком дорого. Для вычисления $k$ главных компонент ($k \ll \min(n,d)$) применяют **усечённый SVD**. В Python доступны:

* `scipy.sparse.linalg.svds` (для плотных и разреженных матриц, использует ARPACK),
* `sklearn.utils.extmath.randomized_svd` — рандомизированный SVD, основанный на случайных проекциях, быстрее для больших $n$.

Реализуем `pca_svd_truncated` с помощью `svds`.

```python
from scipy.sparse.linalg import svds

def pca_svd_truncated(X, k):
    mean = np.mean(X, axis=0)
    Xc = X - mean
    # svds возвращает сингулярные числа в возрастающем порядке, нужно инвертировать
    U, S, Vt = svds(Xc, k=k, which='LM')  # Largest Magnitude
    # Переупорядочиваем по убыванию
    idx = np.argsort(S)[::-1]
    S = S[idx]
    U = U[:, idx]
    Vt = Vt[idx, :]
    X_pca = U @ np.diag(S)
    total_var = np.sum(np.var(Xc, axis=0, ddof=1))  # сумма дисперсий исходных признаков
    explained_variance = (S**2) / (X.shape[0] - 1)
    explained_variance_ratio = explained_variance / total_var
    return X_pca, Vt, explained_variance, explained_variance_ratio, mean
```

Этот метод хорошо работает, когда $k$ мало, а матрица центрированных данных может быть как плотной, так и разреженной. В `sklearn` автоматический выбор (`svd_solver='auto'`) как раз опирается на рандомизированный SVD при большом количестве образцов.

### 7. Выводы и практические рекомендации

1. **Всегда используйте SVD для PCA.** Это более устойчивый и гибкий подход, который не требует явного построения ковариационной матрицы.
2. Если $d \gg n$, SVD через левые сингулярные векторы или усечённый SVD значительно эффективнее.
3. Для больших матриц с $k \ll \min(n,d)$ предпочтителен рандомизированный SVD (`sklearn.decomposition.PCA` с `svd_solver='randomized'`).
4. Стандартизация данных (вычитание среднего и, возможно, масштабирование) остаётся обязательной; SVD выполняется уже на центрированной матрице.
5. Знаки главных компонент произвольны — это свойство собственных векторов, а не недостаток SVD.

**Задание для читателя:**  
Реализуйте функцию PCA через SVD, как показано выше. Сгенерируйте матрицу $500 \times 500$ с элементами из нормального распределения, добавьте небольшой шум. Сравните главные компоненты, полученные вашей функцией `pca_svd` и функцией `pca_eig`. Вычислите максимальное абсолютное расхождение между проекциями и объясните, почему они могут отличаться, несмотря на теоретическую эквивалентность.


## Как выбрать оптимальное число главных компонент?

После того как мы научились вычислять главные компоненты, возникает естественный вопрос: сколько их оставить для дальнейшего анализа? Слишком малое число компонент приведёт к потере существенной информации, слишком большое — сохранит шум и не уменьшит размерность. В этом разделе мы рассмотрим несколько классических подходов к выбору числа $k$, подкрепим их примерами и обсудим границы применимости.

### 1. Кумулятивная объяснённая дисперсия

Каждая главная компонента объясняет долю общей дисперсии, пропорциональную её собственному значению:

$$
\text{explained\_variance\_ratio}_j = \frac{\lambda_j}{\sum_{i=1}^d \lambda_i}.
$$

Накопленная (кумулятивная) доля для первых $k$ компонент:

$$
\text{cumulative\_ratio}(k) = \frac{\sum_{i=1}^k \lambda_i}{\sum_{i=1}^d \lambda_i}.
$$

Часто на практике выбирают наименьшее $k$, при котором кумулятивная доля достигает заданного порога — например, $0.90$ или $0.95$. Это означает, что $k$ главных компонент сохраняют $90\%$ (или $95\%$) вариации, присутствующей в исходных данных.

Построим график «каменистой осыпи» (scree plot), где по оси абсцисс отложен номер компоненты, а по оси ординат — собственное значение $\lambda_j$, и добавим кривую кумулятивной доли.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
import pandas as pd

# Пример: датасет digits (64 признака)
digits = load_digits()
X = digits.data
# Стандартизация не обязательна, но может быть применена;
# для PCA на ковариационной матрице оставим как есть, т.к. признаки одного масштаба
pca_full = PCA().fit(X)
eigenvalues = pca_full.explained_variance_
explained_var_ratio = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained_var_ratio)

fig, ax1 = plt.subplots(figsize=(10,6))
ax1.plot(range(1, len(eigenvalues)+1), eigenvalues, 'b-o', markersize=3, label='Собственное значение')
ax1.set_xlabel('Номер главной компоненты')
ax1.set_ylabel('Собственное значение', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(range(1, len(cumulative)+1), cumulative, 'r-s', markersize=3, label='Кумулятивная доля')
ax2.set_ylabel('Кумулятивная объяснённая дисперсия', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.axhline(y=0.9, color='gray', linestyle='--', label='90% порог')
ax2.axhline(y=0.95, color='gray', linestyle=':', label='95% порог')

plt.title('Scree plot и кумулятивная объяснённая дисперсия (Digits)')
fig.tight_layout()
plt.show()

# Таблица для некоторых k
k_values = [5, 10, 20, 30]
table = pd.DataFrame({
    'k': k_values,
    'Кумулятивная доля': [cumulative[k-1] for k in k_values]
})
print(table)
```

Правило $k=10$ для Digits даёт примерно $0.74$ накопленной дисперсии, $k=20$ — $0.88$, $k=30$ — $0.94$, $k=40$ — $0.97$. Порог $0.95$ достигается при $k\approx 36$.

### 2. Метод «локтя» (elbow)

На графике собственных значений иногда заметен резкий изгиб («локоть»), после которого убывание $\lambda_j$ становится почти линейным. Идея состоит в том, чтобы выбрать $k$, соответствующий точке максимальной кривизны. Хотя метод субъективен, он хорошо работает, когда в данных есть несколько доминирующих направлений вариации, а остальные примерно равны (шум).

Сравним два синтетических случая: один с чётким локтем, другой — с плавным спадом.

```python
np.random.seed(42)
n_samples = 200
# Случай 1: данные с 5 сильными компонентами + шум
X1 = np.random.randn(n_samples, 5) @ np.random.randn(5, 20) + 0.1 * np.random.randn(n_samples, 20)
# Случай 2: данные без выраженной низкоразмерной структуры
X2 = np.random.randn(n_samples, 20)

fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, X, title in zip(axes, [X1, X2], ['Чёткий локоть', 'Плавный спад']):
    pca = PCA().fit(X)
    ax.plot(range(1, len(pca.explained_variance_)+1), pca.explained_variance_, 'o-')
    ax.set_title(title)
    ax.set_xlabel('Номер компоненты')
    ax.set_ylabel('Собственное значение')
plt.tight_layout()
plt.show()
```

В первом случае «локоть» находится примерно на $k=5$, во втором — выделить изгиб затруднительно.

### 3. Правило Кайзера (Kaiser criterion)

Если данные предварительно стандартизированы (каждый признак имеет единичную дисперсию), сумма всех дисперсий равна $d$. Тогда правило Кайзера рекомендует оставлять только те главные компоненты, чьи собственные значения превышают $1$, т.е. $\lambda_j > 1$. Интуитивное обоснование: компонента должна объяснять больше вариации, чем один исходный стандартизированный признак.

Применим к Digits (после стандартизации) и посмотрим, сколько компонент удовлетворяет критерию.

```python
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)
pca_scaled = PCA().fit(X_scaled)
num_kaiser = np.sum(pca_scaled.explained_variance_ > 1)
print(f"Число компонент по правилу Кайзера: {num_kaiser}")
```

Для Digits это число обычно около 12–14. Правило Кайзера особенно полезно в факторном анализе, но в PCA его применение иногда даёт слишком высокое $k$ (поскольку шумовые компоненты могут иметь $\lambda$ чуть больше 1).

### 4. Пермутационный тест

Более статистически обоснованный метод — сравнение наблюдаемых собственных значений с теми, которые можно получить при отсутствии структуры (нулевая гипотеза). Для этого случайно перемешивают значения в каждом столбце независимо, разрушая корреляции между признаками, после чего выполняют PCA на перемешанных данных. Процедуру повторяют много раз и получают распределение наибольшего собственного значения для случайных данных. Компоненты, чьи собственные значения превышают, скажем, 95-й процентиль этого распределения, считаются значимыми. Альтернативно можно сравнивать каждое $\lambda_j$ с собственными значениями из перемешанных матриц.

Реализуем функцию `permutation_test_pca` и применим её к Digits.

```python
def permutation_test_pca(X, n_perm=100, alpha=0.05):
    """
    Пермутационный тест для определения числа значимых компонент.
    Возвращает p-значения для каждого собственного значения.
    """
    n, d = X.shape
    # Стандартизируем данные для корректного сравнения
    X_std = StandardScaler().fit_transform(X)
    pca = PCA().fit(X_std)
    obs_eigvals = pca.explained_variance_
    
    max_eigvals = np.zeros(n_perm)
    # Для каждой перестановки находим наибольшее собственное значение
    for perm in range(n_perm):
        X_perm = np.apply_along_axis(np.random.permutation, axis=0, arr=X_std)
        pca_perm = PCA().fit(X_perm)
        max_eigvals[perm] = pca_perm.explained_variance_[0]
    
    # Сравниваем каждое наблюдаемое собственное значение с распределением максимума
    p_values = np.zeros(d)
    for i in range(d):
        p_values[i] = np.mean(max_eigvals >= obs_eigvals[i])
    # Находим последний индекс, где p-value < alpha
    significant = np.where(p_values < alpha)[0]
    k_perm = significant[-1] + 1 if len(significant) > 0 else 0
    return k_perm, p_values, obs_eigvals, max_eigvals

k_perm, p_vals, obs_eig, max_eig = permutation_test_pca(X, n_perm=100, alpha=0.05)
print(f"Число значимых компонент по пермутационному тесту: {k_perm}")

# Визуализация
plt.figure(figsize=(8,5))
plt.plot(range(1, len(obs_eig)+1), obs_eig, 'b-o', label='Наблюдаемые')
plt.axhline(y=np.percentile(max_eig, 95), color='r', linestyle='--', label='95% процентиль max λ при H0')
plt.xlabel('Номер компоненты')
plt.ylabel('Собственное значение')
plt.title('Пермутационный тест для PCA')
plt.legend()
plt.grid(True)
plt.show()
```

Для Digits пермутационный тест обычно оставляет около 30–40 компонент, что согласуется с высоким порогом кумулятивной дисперсии.

### 5. Анализ на реальном датасете Digits: реконструкция изображений

Помимо численных критериев, полезно визуально оценить качество восстановления данных при уменьшении размерности. Для Digits (изображения 8×8 пикселей) построим реконструкцию цифр при различных $k$.

```python
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

digits = load_digits()
X = digits.data
y = digits.target

# PCA с удержанием k компонент
def reconstruct(pca, X, k):
    # Проекция и обратная реконструкция
    X_pca = pca.transform(X)
    # Оставляем только первые k компонент, остальные зануляем
    X_pca[:, k:] = 0
    return pca.inverse_transform(X_pca)

pca_full = PCA().fit(X)

fig, axes = plt.subplots(4, 5, figsize=(10,8))
k_list = [5, 10, 20, 40]
sample_indices = np.random.choice(len(X), 5, replace=False)

for i, idx in enumerate(sample_indices):
    axes[0, i].imshow(X[idx].reshape(8,8), cmap='gray')
    axes[0, i].set_title(f'Исходная {y[idx]}')
    axes[0, i].axis('off')

for row, k in enumerate(k_list):
    X_rec = reconstruct(pca_full, X, k)
    for col, idx in enumerate(sample_indices):
        axes[row+1, col].imshow(X_rec[idx].reshape(8,8), cmap='gray')
        axes[row+1, col].set_title(f'k={k}')
        axes[row+1, col].axis('off')

plt.suptitle('Реконструкция цифр при разном числе главных компонент')
plt.tight_layout()
plt.show()
```

Визуально при $k=10$ цифры уже узнаваемы, при $k=20$ реконструкция почти неотличима от оригинала, что подтверждает выбор $k \approx 20-30$ как разумный компромисс между сжатием и качеством.

### 6. Когда эти методы не работают

Описанные эвристики могут давать противоречивые результаты в следующих ситуациях:
- Данные не имеют ярко выраженной низкоразмерной структуры (все $\lambda_j$ примерно равны). В этом случае scree plot выглядит как пологий склон без локтя, правило Кайзера может отсеять почти все компоненты, а пермутационный тест — не дать значимых.
- Сильно зашумлённые данные: высокочастотный шум создаёт много компонент с $\lambda_j \approx \text{const}$, что затрудняет выбор.
- Данные с кластерной структурой: PCA ориентирован на глобальные линейные направления, и локтевой метод может не соответствовать оптимальному $k$ для последующей кластеризации.

В таких случаях полезно обратиться к альтернативам.

### 7. Альтернативные методы

**Информационные критерии для вероятностного PCA (PPCA).**  
Вероятностный PCA рассматривает наблюдения как порождённые линейной моделью с латентными переменными: $\mathbf{x} = \mathbf{W}\mathbf{z} + \boldsymbol{\mu} + \boldsymbol{\varepsilon}$, где $\mathbf{z} \sim \mathcal{N}(0,\mathbf{I}_k)$, $\boldsymbol{\varepsilon} \sim \mathcal{N}(0,\sigma^2\mathbf{I}_d)$. Число латентных переменных $k$ можно выбирать, максимизируя байесовский информационный критерий (BIC) или AIC. Реализация доступна в `sklearn.decomposition.PCA` с параметром `svd_solver='full'` и последующим вычислением логарифмического правдоподобия.

**Кросс-валидация ошибки реконструкции.**  
Обучаем PCA на части данных, измеряем среднеквадратичную ошибку восстановления на отложенной выборке. С ростом $k$ ошибка убывает, и можно выбрать $k$, после которого улучшение становится незначительным (аналог локтя на кривой валидации).

**Анализ силуэта или стабильности кластеров.**  
Если PCA используется как предобработка для кластеризации, число компонент можно выбирать, максимизируя качество последующей кластеризации (например, средний силуэт) на валидационных данных. Это нестандартный, но практичный путь.

### Заключение

Выбор числа главных компонент — это компромисс между сохранением информации и сложностью модели. Надёжная стратегия — использовать комбинацию методов: scree plot с кумулятивной дисперсией даёт общую картину; правило Кайзера или пермутационный тест предлагают формальный порог; визуализация реконструкции подтверждает осмысленность выбора. Во многих практических задачах порог $0.90$ или $0.95$ кумулятивной дисперсии является простым и эффективным ориентиром. Помните, что PCA — лишь первый шаг, и окончательное число компонент может диктоваться требованиями конкретного алгоритма или интерпретируемостью.

**Упражнения для читателя:**
1. Почему при стандартизации данных правило Кайзера эквивалентно $\lambda_j > 1$? Что изменится, если данные не стандартизированы?
2. Реализуйте кросс-валидацию ошибки реконструкции для Digits и сравните оптимальное $k$ с кумулятивным порогом $0.95$.
3. Для набора данных `fetch_olivetti_faces` (64x64 изображения лиц) постройте scree plot и определите число компонент, достаточное для визуально приемлемой реконструкции.

### Применения PCA: шумоподавление, визуализация, сжатие и обнаружение аномалий

Метод главных компонент — не просто абстрактная техника снижения размерности. В этом разделе мы рассмотрим его практические применения, каждое из которых демонстрирует силу PCA в реальных задачах. Код будет сопровождать все примеры.

#### 1. Шумоподавление (denoising) с помощью PCA

В данных часто присутствует шум — случайные флуктуации, не несущие полезной информации. Если сигнал сосредоточен в нескольких главных компонентах, а шум распределён примерно равномерно по всем направлениям, то отбрасывание компонент с малыми собственными значениями эквивалентно подавлению шума. Это свойство PCA особенно полезно, когда данные содержат множество шумовых признаков, маскирующих истинную структуру.

**Идея:**
1. Добавить к информативным признакам большое число шумовых.
2. Обучить PCA и оставить столько компонент, сколько было исходных информативных измерений.
3. Восстановить данные по усечённому набору компонент — шум будет значительно ослаблен.

Продемонстрируем на синтетическом примере: двумерные данные `make_moons` с добавлением 20 шумовых признаков. Сравним точность классификации kNN до и после PCA.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Генерируем данные "полумесяцы"
X_signal, y = make_moons(n_samples=300, noise=0.15, random_state=42)
# Добавляем 20 шумовых признаков
noise = np.random.normal(0, 0.5, size=(X_signal.shape[0], 20))
X_noisy = np.hstack([X_signal, noise])

# Разделяем на train/test
X_train, X_test, y_train, y_test = train_test_split(X_noisy, y, test_size=0.3, random_state=0)

# Масштабируем (PCA на нестандартизированных данных с шумом даст доминирование шума)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучаем kNN на зашумлённых данных
knn_noisy = KNeighborsClassifier(n_neighbors=5)
knn_noisy.fit(X_train_scaled, y_train)
acc_noisy = knn_noisy.score(X_test_scaled, y_test)
print(f"Точность на зашумлённых данных: {acc_noisy:.3f}")

# Применяем PCA, оставляя 2 компоненты (истинная размерность)
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# kNN на данных после PCA
knn_pca = KNeighborsClassifier(n_neighbors=5)
knn_pca.fit(X_train_pca, y_train)
acc_pca = knn_pca.score(X_test_pca, y_test)
print(f"Точность после PCA (2 компоненты): {acc_pca:.3f}")

# Визуализация: оригинальные данные (только сигнальная часть) и восстановленные после PCA
X_recovered = pca.inverse_transform(X_test_pca)
# Обратное масштабирование для визуализации только первых двух признаков
X_recovered_2d = scaler.inverse_transform(X_recovered)[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', alpha=0.7)
axes[0].set_title('Зашумлённые тестовые данные (первые 2 признака)')
axes[1].scatter(X_recovered_2d[:, 0], X_recovered_2d[:, 1], c=y_test, cmap='viridis', alpha=0.7)
axes[1].set_title('Восстановленные после PCA')
plt.show()
```

Обычно точность после PCA заметно выше, потому что шум был удалён, а структура «полумесяцев» сохранена (хотя PCA — линейный метод, он может уловить основные направления вариации даже для нелинейных данных).

#### 2. Визуализация многомерных данных: Iris и biplot

PCA часто используется для визуализации высокоразмерных данных в 2D или 3D. Классический пример — датасет Iris (4 признака). Проекция на первые две главные компоненты позволяет увидеть разделение видов, а с помощью **biplot** можно наложить направления исходных признаков, показывая их вклад в компоненты.

```python
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

# Стандартизируем
X_std = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

# Biplot
plt.figure(figsize=(8,6))
for i, target_name in enumerate(iris.target_names):
    plt.scatter(X_pca[y == i, 0], X_pca[y == i, 1], label=target_name, alpha=0.7)

# Добавляем стрелки исходных признаков
loadings = pca.components_.T  # shape (4,2)
for j, name in enumerate(feature_names):
    plt.arrow(0, 0, loadings[j,0]*3, loadings[j,1]*3, color='red', alpha=0.5,
              head_width=0.1, head_length=0.1)
    plt.text(loadings[j,0]*3.2, loadings[j,1]*3.2, name, color='red', fontsize=10)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA визуализация Iris (biplot)')
plt.legend()
plt.grid(True)
plt.show()

# Интерпретация: направление стрелки показывает, с какой компонентой связан признак.
# Например, признак "petal length" имеет большую нагрузку на PC1.
```

Biplot даёт понимание, какие исходные признаки сильнее всего влияют на каждую главную компоненту, что полезно для интерпретации.

#### 3. Сжатие изображений с помощью PCA

Изображение можно рассматривать как матрицу пикселей, где строки — образцы, а столбцы — пиксели (для серого изображения). PCA позволяет сжимать такие данные, сохраняя только несколько компонент и восстанавливая изображение с минимальными потерями. Для цветного изображения PCA можно применить отдельно к каждому каналу или к объединённой матрице.

Загрузим встроенное изображение камеры из `skimage.data`, выполним сжатие и оценим качество метрикой PSNR (пиковое отношение сигнал/шум).

```python
from skimage import data, img_as_float
from skimage.metrics import peak_signal_noise_ratio
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

image = img_as_float(data.camera())  # 512x512
# Для каждого канала (здесь серый, 1 канал) мы можем применить PCA ко всем строкам
# Строки как образцы, столбцы как пиксели.
X_img = image  # (512, 512)
# Центрируем
mean = X_img.mean(axis=0)
X_centered = X_img - mean

# Функция восстановления для заданного k
def pca_compress_reconstruct(X_centered, k):
    pca = PCA(n_components=k)
    X_pca = pca.fit_transform(X_centered)
    X_rec = pca.inverse_transform(X_pca)
    return X_rec + mean, pca

k_values = [10, 20, 50]
fig, axes = plt.subplots(1, 4, figsize=(16,8))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Оригинал')

for ax, k in zip(axes[1:], k_values):
    X_rec, _ = pca_compress_reconstruct(X_centered, k)
    axes[1:][ax.get_geometry()[1]-1].imshow(np.clip(X_rec, 0, 1), cmap='gray')
    axes[1:][ax.get_geometry()[1]-1].set_title(f'k={k}')
    psnr = peak_signal_noise_ratio(image, X_rec, data_range=1)
    print(f'k={k}, PSNR={psnr:.2f} дБ')

plt.tight_layout()
plt.show()
```

При k=50 изображение уже очень близко к оригиналу, при k=20 ещё различимо, но появляются артефакты. PSNR растёт с увеличением числа компонент, отражая улучшение качества.

#### 4. Обнаружение аномалий с помощью ошибки реконструкции

Аномалии — это точки, не соответствующие типичной структуре данных. Если обучить PCA на «нормальных» данных, то аномальные объекты будут плохо восстанавливаться (большая ошибка реконструкции). Этот приём часто используется для детекции выбросов.

Используем синтетический датасет: основная масса из нормального распределения с корреляциями, плюс несколько явных выбросов. Вычислим ошибку реконструкции для каждой точки после PCA с малым числом компонент и установим порог, например, 95-й процентиль.

```python
from scipy.stats import percentileofscore

np.random.seed(123)
# Генерируем "нормальные" данные (100 точек, 5 признаков с корреляциями)
X_normal = np.random.randn(100, 5)
X_normal = X_normal @ np.random.randn(5, 5) + np.array([1, 2, -1, 0.5, 0])
# Добавляем несколько выбросов
X_outlier = np.array([[5, 5, -5, 5, 5], [0, 0, 0, 0, 8], [-3, -3, 3, -3, -3]])
X_data = np.vstack([X_normal, X_outlier])

# Стандартизируем и PCA на 2 компонентах (предполагая, что нормальные данные лежат в низкоразмерном подпространстве)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_data)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
X_rec = pca.inverse_transform(X_pca)

# Ошибка реконструкции (MSE на точку)
reconstruction_error = np.mean((X_scaled - X_rec)**2, axis=1)

# Порог: 95% процентиль на обучающих данных (только нормальные)
threshold = np.percentile(reconstruction_error[:100], 95)
predicted_anomaly = reconstruction_error > threshold

plt.figure(figsize=(8,5))
plt.plot(reconstruction_error, 'o-')
plt.axhline(y=threshold, color='r', linestyle='--', label='Порог (95%)')
for i in range(100, len(X_data)):
    plt.scatter(i, reconstruction_error[i], color='red', s=50, zorder=5, label='Выброс' if i==100 else "")
plt.xlabel('Номер точки')
plt.ylabel('Ошибка реконструкции')
plt.title('Обнаружение аномалий через ошибку реконструкции PCA')
plt.legend()
plt.show()
```

Выбросы обычно чётко выделяются высокими значениями ошибки.

#### 5. Связь с другими методами

PCA — линейный метод. Для нелинейных данных существуют расширения:

- **Kernel PCA**: применяет kernel trick, выполняя PCA в неявно заданном пространстве признаков. Позволяет находить нелинейные многообразия (например, спирали).
- **Sparse PCA**: добавляет L1-регуляризацию к нагрузкам, что делает компоненты разреженными и более интерпретируемыми.
- **Probabilistic PCA (PPCA)**: байесовская модель, где PCA выводится как оценка максимального правдоподобия для линейной модели с латентными переменными. PPCA даёт возможность использовать EM-алгоритм, обрабатывать пропуски и строить доверительные интервалы.

Эти методы расширяют базовый PCA, но его понимание остаётся ключевым.

#### 6. Когда PCA полезен, а когда нет

| Ситуация | Применять PCA? | Обоснование |
|----------|----------------|-------------|
| Данные приблизительно гауссовы, признаки коррелированы | Да | PCA оптимален для гауссовских данных, декоррелирует и снижает размерность |
| Признаки измерены в разных единицах | Да, но с обязательной стандартизацией | Иначе PCA будет захвачен признаками с большим масштабом |
| Данные сильно зашумлены | Да | Шумоподавление отбрасыванием младших компонент |
| Визуализация | Да | Проекция на 2-3 главные компоненты |
| Сжатие с потерями | Да | Сохраняем k компонент, восстанавливаем данные |
| Нелинейная структура (кольца, спирали) | Нет (или Kernel PCA) | Линейные компоненты не разделят нелинейные паттерны |
| Признаки категориальные (после one-hot) | С осторожностью | Разреженность и большая размерность могут сделать PCA неэффективным; лучше использовать TruncatedSVD |
| Кластеризация в высоких размерностях | Да, как предобработка | Снижает шум, ускоряет k-means |

#### Упражнения для читателя

1. Примените PCA к датасету с категориальными признаками после one-hot кодирования. Объясните, почему результат может быть непригоден для интерпретации и как с этим бороться (например, использовать MCA — множественный анализ соответствий).
2. Почему перед PCA необходимо центрировать данные, но не всегда нужно масштабировать их до единичной дисперсии? Приведите пример, когда масштабирование обязательно, и пример, когда оно вредно.
3. Реализуйте простой детектор аномалий на основе PCA для датасета `load_wine()`. Визуализируйте точки с наибольшей ошибкой реконструкции на графике первых двух главных компонент.
